# Statsapp YOLO overnight finetune

Unattended Kaggle **GPU** train from your existing `pretrained_best.pt`.

## Before you leave
1. Settings → **Accelerator = GPU**
2. Settings → **Internet ON**
3. Add Input → `yohandeshpande/soccernet5-yolo`
4. **Save Version → Save & Run All**

## After
Download `/kaggle/working/best.pt` → `Desktop/statsapp/models/best.pt`


In [ ]:
# 0) Overnight config
EPOCHS = 50
IMGSZ = 960
BATCH = 8          # auto-falls to 4 on OOM
PATIENCE = 20
SAVE_PERIOD = 1    # checkpoint every epoch
PROJECT = "/kaggle/working/runs"
NAME = "kaggle_finetune_v9c"
SEED = 42


In [ ]:
# 1) GPU check
import torch
print("cuda:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise SystemExit("GPU OFF. Settings → Accelerator → GPU, then restart & Save Version again.")
for i in range(torch.cuda.device_count()):
    print(f"  gpu[{i}]:", torch.cuda.get_device_name(i))
DEVICE = 0 if torch.cuda.device_count() == 1 else [0, 1]
print("train device:", DEVICE)


In [ ]:
# 2) Install ultralytics
import subprocess, sys
from pathlib import Path

def sh(cmd):
    print(">>", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

try:
    sh([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
except Exception as e:
    print("online pip failed:", e, flush=True)
    wheels = Path("/kaggle/input/ultralytics-wheels")
    if not wheels.exists():
        hits = list(Path("/kaggle/input").rglob("ultralytics*.whl"))
        if not hits:
            raise SystemExit("pip failed and ultralytics-wheels not attached. Enable Internet or add that dataset.")
        wheels = hits[0].parent
    sh([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", f"--find-links={wheels}",
        "ultralytics", "ultralytics-thop", "nvidia-ml-py",
    ])

import ultralytics
print("ultralytics", ultralytics.__version__, flush=True)


In [ ]:
# 3) Find dataset + rewrite data.yaml to the REAL Kaggle mount
from pathlib import Path
import yaml

root = Path("/kaggle/input")
print("=== /kaggle/input ===", flush=True)
if not root.exists() or not any(root.iterdir()):
    raise SystemExit("/kaggle/input EMPTY. Add Input soccernet5-yolo and Save Version again.")
for p in sorted(root.iterdir()):
    print(" ", p.name, flush=True)

ymls = sorted(root.rglob("data.yaml"))
print("data.yaml hits:", [str(x) for x in ymls], flush=True)
if not ymls:
    raise SystemExit("No data.yaml under /kaggle/input")

prefer = [p for p in ymls if "soccernet5" in str(p)]
src_yaml = prefer[0] if prefer else ymls[0]
ds_root = src_yaml.parent  # folder that actually has train/valid/test

pts = list(ds_root.glob("pretrained_best.pt")) + list(ds_root.glob("best.pt"))
if not pts:
    pts = list(root.rglob("pretrained_best.pt")) + list(root.rglob("best.pt"))
if not pts:
    raise SystemExit("No pretrained_best.pt/best.pt found")
pref_w = [p for p in pts if p.name == "pretrained_best.pt"]
WEIGHTS = str((pref_w[0] if pref_w else pts[0]).resolve())

for split in ("train", "valid", "test"):
    img = ds_root / split / "images"
    print(f"{split}:", img, "exists=" + str(img.is_dir()), flush=True)
if not (ds_root / "train" / "images").is_dir():
    raise SystemExit(f"Missing train/images under {ds_root}")
if not (ds_root / "valid" / "images").is_dir():
    raise SystemExit(f"Missing valid/images under {ds_root}")

# CRITICAL: uploaded data.yaml had path=/kaggle/input/soccernet5-yolo but
# Kaggle often mounts at /kaggle/input/datasets/yohandeshpande/soccernet5-yolo
cfg = yaml.safe_load(src_yaml.read_text()) or {}
cfg["path"] = str(ds_root.resolve())
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg["test"] = "test/images"
cfg["nc"] = int(cfg.get("nc", 5))
if "names" not in cfg:
    cfg["names"] = {0: "ball", 1: "player", 2: "goalkeeper", 3: "referee", 4: "goalpost"}

DATA = "/kaggle/working/data.yaml"
Path(DATA).write_text(yaml.safe_dump(cfg, sort_keys=False))
print("DATA   =", DATA, flush=True)
print("WEIGHTS=", WEIGHTS, flush=True)
print("ds_root=", ds_root, flush=True)
print("--- rewritten data.yaml ---", flush=True)
print(Path(DATA).read_text(), flush=True)
print("dataset OK", flush=True)


In [ ]:
# 4) Train overnight (resume if last.pt exists)
from pathlib import Path
from ultralytics import YOLO
import shutil
import torch

run_dir = Path(PROJECT) / NAME
last_pt = run_dir / "weights" / "last.pt"
best_pt = run_dir / "weights" / "best.pt"
out_best = Path("/kaggle/working/best.pt")
out_last = Path("/kaggle/working/last.pt")

resume = last_pt.is_file()
if resume:
    print("RESUMING from", last_pt, flush=True)
    model = YOLO(str(last_pt))
else:
    print("STARTING from pretrained", WEIGHTS, flush=True)
    model = YOLO(WEIGHTS)

def train_once(batch, resume_flag):
    return model.train(
        data=DATA,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=batch,
        device=DEVICE,
        project=PROJECT,
        name=NAME,
        exist_ok=True,
        resume=resume_flag,
        patience=PATIENCE,
        workers=2,
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        cos_lr=True,
        close_mosaic=10,
        seed=SEED,
        deterministic=False,
        amp=True,
        save=True,
        save_period=SAVE_PERIOD,
        plots=True,
        verbose=True,
    )

batch = BATCH
try:
    results = train_once(batch, resume)
except RuntimeError as e:
    msg = str(e).lower()
    if "out of memory" in msg or ("cuda" in msg and "memory" in msg):
        print("OOM at batch", batch, "→ retry batch 4", flush=True)
        torch.cuda.empty_cache()
        batch = 4
        resume2 = (run_dir / "weights" / "last.pt").is_file()
        model = YOLO(str(run_dir / "weights" / "last.pt")) if resume2 else YOLO(WEIGHTS)
        results = train_once(batch, resume2)
    else:
        raise

if best_pt.is_file():
    shutil.copy2(best_pt, out_best)
    print("copied", best_pt, "→", out_best, flush=True)
if last_pt.is_file():
    shutil.copy2(last_pt, out_last)
    print("copied", last_pt, "→", out_last, flush=True)

print("DONE", flush=True)
print("download:", out_best if out_best.is_file() else best_pt, flush=True)


## Overnight notes
- Checkpoints every epoch → re-run the same notebook to resume from `last.pt` if it dies.
- Final file: `/kaggle/working/best.pt`
- Cell 3 rewrites `data.yaml` so image paths match the real mount (`/kaggle/input/datasets/...`).
